In [2]:
# =========================
# 5.1 Data Pre-processing
# =========================

import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# =========================
# 1. Load Datasets
# =========================
users_train = pd.read_csv("/content/train_users.csv")
users_test  = pd.read_csv("/content/test_users.csv")
articles    = pd.read_csv("/content/news_articles.csv")

print("Train users:", users_train.shape)
print("Test users :", users_test.shape)
print("Articles   :", articles.shape)

# =========================
# 2. USER DATA PREPROCESSING
# =========================
# Target = user context
TARGET_COL = "label"

X_user = users_train.drop(columns=[TARGET_COL])
y_user = users_train[TARGET_COL]

# Identify feature types
num_cols = X_user.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_user.select_dtypes(include=["object"]).columns

# Pipelines
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

user_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, num_cols),
        ("cat", categorical_pipeline, cat_cols)
    ]
)

# Fit ONLY on training users
X_user_processed = user_preprocessor.fit_transform(X_user)

print("Processed user feature shape:", X_user_processed.shape)

# Save user preprocessor
joblib.dump(user_preprocessor, "user_preprocessor.pkl")

# =========================
# 3. ARTICLE DATA CLEANING
# =========================
# Drop articles with missing category (cannot be assigned to an arm)
articles_clean = articles.dropna(subset=["category"]).reset_index(drop=True)

print("Articles after cleaning:", articles_clean.shape)

# Group articles by category for recommendation
articles_by_category = {
    category: df.reset_index(drop=True)
    for category, df in articles_clean.groupby("category")
}

print("Available article categories:", list(articles_by_category.keys()))

# Save cleaned articles
joblib.dump(articles_by_category, "articles_by_category.pkl")

print("\nData preprocessing complete.")


Train users: (2000, 6)
Test users : (2000, 6)
Articles   : (16869, 6)
Processed user feature shape: (2000, 5)
Articles after cleaning: (16868, 6)
Available article categories: ['ARTS & CULTURE', 'BLACK VOICES', 'BUSINESS', 'COLLEGE', 'COMEDY', 'CRIME', 'CULTURE & ARTS', 'EDUCATION', 'ENTERTAINMENT', 'ENVIRONMENT', 'FOOD & DRINK', 'GREEN', 'HEALTHY LIVING', 'HOME & LIVING', 'IMPACT', 'LATINO VOICES', 'MEDIA', 'MONEY', 'PARENTING', 'PARENTS', 'POLITICS', 'QUEER VOICES', 'RELIGION', 'SCIENCE', 'SPORTS', 'STYLE', 'STYLE & BEAUTY', 'TASTE', 'TECH', 'TRAVEL', 'U.S. NEWS', 'WEDDINGS', 'WEIRD NEWS', 'WELLNESS', 'WOMEN', 'WORLD NEWS']

Data preprocessing complete.


In [3]:
# =========================
# 5.2 User Classification (Decision Tree)
# =========================

import pandas as pd
import joblib

from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# =========================
# 1. Load Data
# =========================
train_users = pd.read_csv("/content/train_users.csv")
test_users  = pd.read_csv("/content/test_users.csv")

TARGET_COL = "label"

X_train = train_users.drop(columns=[TARGET_COL])
y_train = train_users[TARGET_COL]

X_test = test_users.drop(columns=[TARGET_COL])
y_test = test_users[TARGET_COL]

# =========================
# 2. Load Preprocessor
# =========================
user_preprocessor = joblib.load("user_preprocessor.pkl")

# =========================
# 3. Decision Tree Model
# =========================
dt_classifier = DecisionTreeClassifier(
    max_depth=6,            # prevents overfitting
    min_samples_leaf=20,    # smoother splits
    random_state=42
)

user_classifier = Pipeline(steps=[
    ("preprocessor", user_preprocessor),
    ("classifier", dt_classifier)
])

# =========================
# 4. Train
# =========================
user_classifier.fit(X_train, y_train)

# =========================
# 5. Evaluate
# =========================
y_pred = user_classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\nDecision Tree Accuracy: {accuracy:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

# =========================
# 6. Save Model
# =========================
joblib.dump(user_classifier, "user_context_classifier_dt.pkl")

print("Decision Tree classifier saved as user_context_classifier_dt.pkl")



Decision Tree Accuracy: 0.3465

Classification Report:
              precision    recall  f1-score   support

       user1       0.35      0.67      0.46       672
       user2       0.34      0.08      0.12       679
       user3       0.34      0.29      0.31       649

    accuracy                           0.35      2000
   macro avg       0.34      0.35      0.30      2000
weighted avg       0.34      0.35      0.30      2000

Decision Tree classifier saved as user_context_classifier_dt.pkl
